# Lesson 01 - Introduktion till AI-agenter

Välkommen till den första lektionen i kursen **AI-agenter för nybörjare**!

En **AI-agent** är ett program som använder en stor språkmodell (LLM) som sin resonemangsmotor och kan vidta *åtgärder* i verkliga världen — anropa API:er, fråga databaser eller köra kod — för att uppnå ett mål på uppdrag av en användare.

I denna anteckningsbok kommer du att bygga din första agent: en **reseagent** som rekommenderar semesterresmål. Under vägen kommer du att lära dig hur man:

1. Ansluter till Azure AI Foundry Agent Service med hjälp av **Microsoft Agent Framework**.
2. Ger agenten ett **verktyg** — en vanlig Python-funktion den kan anropa.
3. Kör agenten och undersöker dess svar.
4. Streamar agentens svar token för token.


## Installation

Innan du kör denna anteckningsbok, se till att du har:

1. **Ett Azure AI Foundry-projekt** med en distribuerad chattmodell (t.ex. `gpt-4o-mini`).
2. **Loggat in med Azure CLI** — kör `az login` i din terminal.
3. **Satt de nödvändiga miljövariablerna:**
   - `AZURE_AI_PROJECT_ENDPOINT` — din Azure AI Foundry-projekts slutpunkt.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — namnet på din distribuerade modell.

Cellen nedan installerar de Python-paket du behöver.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Skapa din första agent

En agent behöver två saker:

- **Instruktioner** som berättar *vem den är* och *hur den ska bete sig* (en systemuppmaning).
- **Verktyg** — Python-funktioner dekorerade med `@tool` som agenten kan anropa för att hämta information eller utföra åtgärder.

Nedan definierar vi ett enkelt verktyg som returnerar en lista över populära semesterdestinationer. Agenten kommer att använda detta verktyg när en användare efterfrågar resetips.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Responses

För en mer interaktiv upplevelse kan du **streama** agentens svar. Istället för att vänta på hela svaret levererar agenten textdelar allteftersom de genereras. Detta är särskilt användbart i chattgränssnitt där du vill visa output i realtid.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Sammanfattning

I denna lektion lärde du dig hur du:

- **Skapar en leverantör** som ansluter till Azure AI Foundry Agent Service via `AzureAIProjectAgentProvider`.
- **Definierar ett verktyg** med hjälp av `@tool` dekoratorn så att agenten kan anropa dina Python-funktioner.
- **Kör agenten** med ett användarmeddelande och skriver ut dess svar.
- **Strömmar svar** för realtidsutdata.

I nästa lektion kommer vi att utforska agent-baserade ramverk mer i detalj och lära oss hur vi ger agenter mer kraftfulla verktyg och förmåga till flerstegsresonemang.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, var vänlig notera att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål bör betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår till följd av användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
